# 00 · Setup check

Run this once per new environment, before spending GPU hours. **≈ 5–10 minutes.**

**Notebook settings (right-hand panel):** Accelerator **GPU T4 ×2** (or P100) · Internet **On** · Persistence *off* is fine.

**Inputs to attach (Add Input):**
1. your private dataset with this code (e.g. `oodlab-code`) — GUIDE.md §2
2. Competition **ImageNet Object Localization Challenge** (accept its rules on the competition page first)

Every check prints PASS/FAIL and the notebook keeps going, so you see *all* problems in one run.

In [ ]:
# ── Set-up 1/2: find the oodlab code (attached as a private Kaggle dataset) ─────────
import os, sys

def find_oodlab(roots, max_depth=4):
    """Bounded breadth-first search for a folder containing oodlab/__init__.py.
    Never walks image folders (the old notebook hung on a recursive glob over ImageNet)."""
    skip = {"ILSVRC", "train", "val", "test", "images", "__pycache__", "cache", "results", "logs"}
    for root in [r for r in roots if r and os.path.isdir(r)]:
        frontier = [(root, 0)]
        while frontier:
            d, depth = frontier.pop(0)
            if os.path.isfile(os.path.join(d, "oodlab", "__init__.py")):
                return d
            if depth < max_depth:
                try:
                    subs = sorted(e.path for e in os.scandir(d)
                                  if e.is_dir(follow_symlinks=False) and e.name not in skip)
                except OSError:
                    continue
                frontier += [(s, depth + 1) for s in subs]
    return None

ROOTS = [os.environ.get("OODLAB_CODE"), "/kaggle/input", "/kaggle/working", os.getcwd()]
CODE_ROOT = find_oodlab(ROOTS)
if CODE_ROOT is None:   # the dataset was uploaded as a zip that Kaggle did not unpack -> unpack it ourselves
    import zipfile
    for root in [r for r in ROOTS[:2] if r and os.path.isdir(r)]:
        for dp, dn, fns in os.walk(root):
            dn[:] = [d for d in dn if d not in {"ILSVRC", "train", "val", "test", "images"}] if dp.count(os.sep) - root.count(os.sep) < 3 else []
            for fn in fns:
                if fn.startswith("oodlab") and fn.endswith(".zip"):
                    zipfile.ZipFile(os.path.join(dp, fn)).extractall("/kaggle/working/_oodlab_code")
    CODE_ROOT = find_oodlab(["/kaggle/working/_oodlab_code"])
if CODE_ROOT is None:
    raise FileNotFoundError("oodlab code not found: attach your private dataset with the code "
                            "(Add Input -> Your Datasets -> oodlab-code). See GUIDE.md, section 2.")
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)
import oodlab
print("oodlab", oodlab.__version__, "loaded from", CODE_ROOT)

In [ ]:
# ── Set-up 2/2: packages, folders, logging ─────────────────────────────────────────
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True, gdown=True)      # CLIP ships inside oodlab; this only adds small pip packages if missing
ctx = start("00_setup_check", need_gpu=False)
log = ctx.log                                   # log.info(...) -> screen + /kaggle/working/logs/00_setup_check.log
import numpy as np, pandas as pd, torch, time, json
from oodlab.report import show, save_tables, write_report

In [ ]:
import traceback, subprocess
checks = []

def check(name, fn):
    t0 = time.time()
    try:
        detail = fn()
        checks.append({"check": name, "status": "PASS", "detail": str(detail)[:300], "sec": round(time.time() - t0, 1)})
    except Exception as e:
        log.error(traceback.format_exc())
        checks.append({"check": name, "status": "FAIL", "detail": f"{type(e).__name__}: {e}"[:300], "sec": round(time.time() - t0, 1)})
    print(f"[{checks[-1]['status']}] {name}: {checks[-1]['detail']}")

### 1 · GPU and disk

In [ ]:
def gpu():
    if ctx.smoke:
        return "skipped in smoke mode"
    assert torch.cuda.is_available(), "no GPU: Settings -> Accelerator -> GPU"
    p = torch.cuda.get_device_properties(0)
    return f"{torch.cuda.device_count()} x {p.name}, {p.total_memory / 1e9:.1f} GB, torch {torch.__version__}"
check("GPU", gpu)

def disk():
    import shutil
    free = {d: shutil.disk_usage(d).free / 1e9 for d in [ctx.dirs.work, ctx.dirs.scratch]}
    msg = ", ".join(f"{k}: {v:.0f} GB free" for k, v in free.items())
    assert ctx.smoke or free[ctx.dirs.scratch] > 25, "need > 25 GB of scratch space for the OpenOOD downloads"
    return msg
check("disk space", disk)

### 2 · Unit tests (synthetic data, CPU) and GPU equivalence with the official code

The tests run the **vendored official** TANL / AdaNeg / NegLabel postprocessors and our
re-implementations on the same stream and require identical results (≈1e-6 in fp32, bit-exact in fp16).

In [ ]:
def unit_tests():
    try:
        import pytest  # noqa: F401
    except ImportError:
        from oodlab.session import pip_install
        pip_install("pytest")
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider",
                        os.path.join(CODE_ROOT, "tests")], capture_output=True, text=True, cwd=ctx.dirs.scratch)
    out = (r.stdout + r.stderr).strip().splitlines()
    assert r.returncode == 0, "\n".join(out[-40:])
    return out[-1]
check("unit tests", unit_tests)

def gpu_equivalence():
    from oodlab.synthetic import make_bank, make_images
    from oodlab.methods import TANLConfig, AdaNegConfig
    from oodlab.official.equivalence import compare_tanl, compare_adaneg
    dev = ctx.device
    b = make_bank(C=100, N=5000, D=512, n_neglabel=1000)
    fi, _ = make_images(b, 3000, "id", seed=1); fo, _ = make_images(b, 2000, "ood", seed=2)
    x = torch.cat([fo, fi])[torch.randperm(5000, generator=torch.Generator().manual_seed(0))]
    batches = list(x.split(256))
    cfg = TANLConfig.paper().with_(num_neg=300, queue_len=100)
    r32 = compare_tanl(b.id_text, b.corpus, b.noise, batches, 1000, cfg, device=dev)
    h = lambda t: t.half()
    r16 = compare_tanl(h(b.id_text), h(b.corpus), h(b.noise), [h(t) for t in batches], 1000,
                       cfg.with_(emulate_fp16=True, exact_fp16_final=True), device=dev)
    a16 = compare_adaneg(h(b.id_text), h(b.corpus[:1000]), [h(t) for t in batches], AdaNegConfig(emulate_fp16=True), device=dev)
    # fp16: bit-identical is expected; allow ~1 fp16 ulp in case a GPU picks another kernel
    ok = (r32["max_abs_diff"] < 1e-4 and r16["max_abs_diff"] <= 2e-3 and a16["max_abs_diff"] <= 2e-3
          and min(r32["pred_agreement"], r16["pred_agreement"], a16["pred_agreement"]) >= 0.999)
    assert ok, (r32, r16, a16)
    exact = "bit-identical" if r16["max_abs_diff"] == 0 and a16["max_abs_diff"] == 0 else "within 1 fp16 ulp"
    return f"{dev}: TANL fp32 max diff {r32['max_abs_diff']:.1e}; fp16 TANL/AdaNeg {exact}"
check("official == ours on " + ctx.device, gpu_equivalence)

### 3 · ImageNet from the Kaggle competition

Kaggle's validation folder is flat; labels come from `LOC_val_solution.csv`. The class order is
checked against OpenOOD-VLM's (`all_wnids`), so label *i* ↔ class name *i* of the text prompts.

In [ ]:
from oodlab.data.imagenet import locate_imagenet, imagenet_val_manifest
state = {}

def imagenet():
    loc = locate_imagenet(ctx.input_roots)
    m = imagenet_val_manifest(loc, expected=None if ctx.smoke else 50000)
    miss = m.missing_files(sample=300)
    assert not miss, f"missing files, e.g. {miss[:3]}"
    state["imagenet"] = m
    return f"{loc.base} ({loc.layout}), {len(m)} images, {len(set(m.labels.tolist()))} classes"
check("ImageNet val on Kaggle", imagenet)

### 4 · CLIP ViT-B/16 and a zero-shot sanity check

Loads CLIP exactly like the official code (fp16 on GPU) and classifies 1,000 random validation
images with the prompt *"The nice {class}."*. Expect roughly **64–70 %** top-1 (ViT-B/16 zero-shot
is ≈ 67 % on the full set; 1,000 images give ± 3 %).

In [ ]:
from oodlab.clipwrap import load_clip, label_text_features, TEMPLATES, build_transform, encode_image_batch, load_image

def clip_check():
    model = load_clip("ViT-B/16", device=ctx.device, input_roots=ctx.input_roots, random_init=ctx.smoke)
    state["model"] = model
    ls = model.logit_scale.exp().item()
    return f"dtype {model.dtype}, input {model.visual.input_resolution}px, logit scale {ls:.6f}"
check("CLIP ViT-B/16 loads", clip_check)

def zero_shot():
    model, m = state["model"], state["imagenet"]
    text = label_text_features(model, oodlab.imagenet_classnames(), TEMPLATES["nice"])
    idx = np.random.default_rng(0).choice(len(m), size=min(1000, len(m)), replace=False)
    sub = m.subset(idx)
    tf = build_transform(model.visual.input_resolution, round(model.visual.input_resolution * 256 / 224))

    class DS(torch.utils.data.Dataset):
        def __len__(self): return len(sub)
        def __getitem__(self, i): return tf(load_image(sub.paths[i])), int(sub.labels[i])

    dl = torch.utils.data.DataLoader(DS(), batch_size=250, num_workers=0 if ctx.smoke else 4)
    correct, n, t0 = 0, 0, time.time()
    for x, y in dl:
        f = encode_image_batch(model, x)
        correct += int(((f @ text.t()).argmax(1).cpu() == y).sum()); n += len(y)
    acc = 100 * correct / n
    rate = n / (time.time() - t0)
    assert ctx.smoke or 60 <= acc <= 75, f"zero-shot top-1 {acc:.1f}% is outside 60-75%: check labels/transform"
    return f"top-1 {acc:.1f}% on {n} images, {rate:.0f} img/s (decode + encode)"
check("zero-shot accuracy", zero_shot)

### 5 · Google Drive (OpenOOD downloads)

Downloads OpenOOD's small image-list archive. If this fails with a quota message, Drive is
rate-limiting the file: use the manual route in GUIDE.md §5 (upload the zips as a Kaggle dataset).

In [ ]:
def drive():
    if ctx.smoke:
        return "skipped in smoke mode"
    from oodlab.data.openood import gdown_zip
    import zipfile
    z = gdown_zip("benchmark_imglist", os.path.join(ctx.dirs.scratch, "zips"), retries=2, wait_s=10, logger=log)
    names = zipfile.ZipFile(z).namelist()
    assert any(n.endswith("imagenet/test_ninco.txt") for n in names), "unexpected archive content"
    return f"{os.path.getsize(z) / 1e6:.1f} MB, {len(names)} entries"
check("Google Drive download", drive)

### Summary

In [ ]:
table = pd.DataFrame(checks)
show(table, "setup checks")
table.to_csv(ctx.path("checks.csv"), index=False)
n_fail = int((table.status == "FAIL").sum())
finish(ctx, {"failed_checks": n_fail})
print("ALL CHECKS PASSED - continue with notebook 01" if n_fail == 0 else f"{n_fail} check(s) FAILED - see GUIDE.md §8 (troubleshooting)")